# Two Signals, One Event: Spectral and Semantic Analysis of Earnings Announcements

**Author:** Nikola Kolev  
**Course:** Data Science, SoftUni 2026  
**Repository:** https://github.com/niki7o/earnings-signals

---

## Abstract

Earnings announcements are the most concentrated moments of information 
revelation in financial markets. At these events, two fundamentally different 
types of signals exist simultaneously: a **frequency-domain signal** derived 
from pre-announcement price dynamics, and a **semantic signal** derived from 
management speech during the announcement itself.

This project investigates three pre-registered hypotheses:

- **H₁:** Spectral entropy of pre-earnings price dynamics predicts 
  post-announcement abnormal returns
- **H₂:** Loughran-McDonald sentiment in earnings call transcripts predicts 
  post-announcement abnormal returns  
- **H₃:** The two signals are statistically independent — they observe 
  different aspects of the same event

All hypotheses are tested on 123 matched earnings events across 15 S&P 500 
technology companies (2019–2023), using Bonferroni-corrected significance 
thresholds to control for multiple comparisons.

---
## 1. Problem Formulation and Significance

### 1.1 The Efficient Market Hypothesis

The Efficient Market Hypothesis (EMH), formalized by Fama (1970), states that 
asset prices reflect all available information. In its **semi-strong form**, 
prices incorporate all publicly available information — including earnings 
announcements — instantaneously upon release.

If EMH holds, no publicly available signal should systematically predict 
post-announcement returns. This gives us a theoretical prediction **before 
we look at any data**: H₁ and H₂ should both fail to reject the null hypothesis.

This is not a weakness of our study design — it is a testable prediction. 
A project that confirms EMH empirically on real data is scientifically more 
honest than one that data-mines until it finds spurious correlations.

### 1.2 Why Two Signals?

Previous studies have examined price-based signals and text-based signals 
separately. The originality of this project lies in H₃: **testing whether 
the two signals are independent of each other**.

If independent, they observe different aspects of the information environment 
and could in principle be combined. If correlated, one is redundant.

### 1.3 Data Sources

This project uses two genuinely independent data sources:

| Source | Type | Provider | Coverage |
|--------|------|----------|----------|
| Daily OHLCV prices | Numerical time series | Yahoo Finance via `yfinance` | 2018–2024 |
| Earnings call transcripts | Unstructured text | Motley Fool via Kaggle | 2019–2023 |

The sources differ in origin, format, collection method, and information type. 
Merging them on ticker and date is a non-trivial data engineering task 
documented in Section 3.

---
## 2. Mathematical Foundation

### 2.1 Log Returns

Raw prices are non-stationary — they trend over time, which violates the 
assumptions of Fourier analysis. We transform prices to **log returns**:

$$r_t = \log\left(\frac{P_t}{P_{t-1}}\right)$$

Log returns are approximately stationary, additive over time, and 
symmetrically distributed around zero. This makes them suitable as 
input to the DFT.

### 2.2 Discrete Fourier Transform

Given a time series of log returns $r[0], r[1], \ldots, r[N-1]$, 
the Discrete Fourier Transform decomposes it into frequency components:

$$X[k] = \sum_{n=0}^{N-1} r[n] \cdot e^{-2\pi i k n / N}, \quad k = 0, 1, \ldots, N-1$$

Each coefficient $X[k]$ represents the amplitude and phase of a sinusoidal 
component at frequency $f_k = k/N$ cycles per trading day.

We use the **Fast Fourier Transform (FFT)** algorithm, which computes the 
DFT in $\mathcal{O}(N \log N)$ rather than $\mathcal{O}(N^2)$.

### 2.3 Power Spectral Density

The Power Spectral Density (PSD) measures how much energy exists at each frequency:

$$P[k] = \frac{|X[k]|^2}{\sum_{j=0}^{N-1} |X[j]|^2}$$

Normalizing by total power ensures $\sum_k P[k] = 1$, making $P$ 
interpretable as a probability distribution over frequencies.

### 2.4 Spectral Entropy

Borrowing from information theory, we define **spectral entropy** as the 
Shannon entropy of the normalized PSD:

$$H_{spec} = -\sum_{k=0}^{N/2} P[k] \log P[k]$$

**Interpretation:**
- $H_{spec} \rightarrow \log(N/2)$ (maximum): energy is uniformly spread 
  across all frequencies — the price series is maximally disordered (noise)
- $H_{spec} \rightarrow 0$ (minimum): energy concentrates at one frequency — 
  the price has a dominant rhythm

For our 30-day window, the theoretical maximum is $\log(16) \approx 2.77$.

### 2.5 Abnormal Returns

To isolate company-specific price movements from market-wide movements, 
we compute **abnormal returns**:

$$AR_t = r_{\text{stock},t} - r_{\text{benchmark},t}$$

where the benchmark is the S&P 500 ETF (SPY). The **cumulative abnormal 
return** over the 5-day post-announcement window is:

$$CAR = \sum_{t=0}^{4} AR_t$$

### 2.6 Loughran-McDonald Sentiment

General-purpose sentiment tools fail on financial text. The word "liability" 
is negative in finance but neutral in everyday English. The 
**Loughran-McDonald (LM) dictionary** was constructed specifically for 
SEC filings and earnings calls.

The LM polarity score is defined as:

$$S_{LM} = \frac{N_{pos} - N_{neg}}{N_{pos} + N_{neg}}$$

where $N_{pos}$ and $N_{neg}$ are counts of positive and negative stems 
found in the transcript. The score ranges from $-1$ (entirely negative) 
to $+1$ (entirely positive).

### 2.7 Hypothesis Testing Framework

All tests use a **Bonferroni-corrected significance threshold** to control 
the family-wise error rate across three simultaneous tests:

$$\alpha_{corrected} = \frac{\alpha}{m} = \frac{0.05}{3} \approx 0.0167$$

where $m = 3$ is the number of hypotheses. This ensures that the 
probability of at least one false positive across all three tests 
remains below 5%.

**Effect sizes** are reported using Cohen's $d$:

$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_{pooled}}, \quad 
s_{pooled} = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}}$$

Interpretation: $|d| < 0.2$ negligible, $< 0.5$ small, $< 0.8$ medium, 
$\geq 0.8$ large.

---
## 3. Data Pipeline

### 3.1 Universe Selection

We fix our analysis universe **before examining any data** to prevent 
selection bias. The universe consists of 15 S&P 500 technology sector 
companies chosen for data availability and transcript quality:

`AAPL, MSFT, GOOGL, AMZN, META, NVDA, AMD, INTC, TSLA, NFLX, ORCL, CRM, ADBE, QCOM, TXN`

**Analysis period:** 2019-01-01 to 2023-12-31  
**Excluded:** Q2 2020 (2020-03-15 to 2020-06-30) — VIX exceeded 80, 
circuit breakers were triggered, and Federal Reserve emergency interventions 
created structural breaks that would contaminate spectral analysis.

### 3.2 Event Date Alignment

A critical methodological decision: earnings calls occur **after market 
close** (~5PM ET). Therefore:

- The closing price on the day of the call already reflects pre-call information
- The first price that reflects the announcement is the **next day's open**

We define the event date $T$ as the first trading day after the earnings call:

Pre and post windows are strictly non-overlapping — enforced in code 
with a `<` comparison on the event date.

### 3.3 Data Sources and Merging

**Source 1 — Price data:** Downloaded via `yfinance` for all 15 tickers 
plus SPY (benchmark). Cached locally as CSV to ensure reproducibility.

**Source 2 — Transcripts:** Motley Fool earnings call dataset (18,755 
transcripts). Filtered to our universe and analysis window.

**Merging challenge:** The two sources use different date formats and 
granularities. Transcripts are matched to earnings dates within a 
±3 day tolerance to account for calendar differences.

**Result:** 210 earnings events from Yahoo Finance × 274 transcripts 
from Motley Fool → **123 matched events** (58.6% match rate).

The 87 unmatched events are concentrated in 2023 Q2–Q4, consistent 
with the Motley Fool dataset's scrape date of May 2023. Missing data 
is therefore not random with respect to company or market conditions — 
it is a temporal cutoff. This is documented as a limitation in Section 6.